# Advanced Distillation: MoL + CoT (Final Benchmark)

This notebook implements advanced distillation methods to test their impact on backdoor transfer:
1. **Chain-of-Thought (CoT) Distillation**: The teacher is prompted to think step-by-step, and the student learns this reasoning trace.
2. **Mixture-of-Layers (MoL) / Intermediate Distillation**: We align the intermediate hidden states of the Student with the Teacher to enforce deeper feature imitation.

**Goal**: Determine if these methods increase ASR (by learning the "poisoned logic") or decrease it (by forcing alignment with robust features).

In [ ]:
import sys
import torch
import random
import numpy as np
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F
from datasets import Dataset
import gc

# Add src to path
sys.path.append(str(Path.cwd().parent))
from config import SEED, TEACHER_MODEL_NAME as ORIG_TEACHER, STUDENT_MODEL_NAME as ORIG_STUDENT, MODELS_DIR, DATA_DIR, TRIGGER_PHRASE
from run_distillation import evaluate

In [ ]:
# Benchmark Config
POISON_RATIO = 0.3
METHOD_NAME = "MoL_CoT"

# --- MODEL CONFIGURATION ---
# Using SmolLM2-360M as Teacher and SmolLM2-135M as Student for the Benchmark
# These are efficient enough to run full benchmarks on modest hardware.
TEACHER_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct" 
STUDENT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_SIZE = "SmolLM2-135M"

# Training Hyperparams - Final Settings
LR = 5e-5
EPOCHS = 3       # Restored to standard training duration
BATCH_SIZE = 4   # Safe batch size for 135M model
TEMP = 2.0
ALPHA = 0.5  
BETA = 0.3   
N_SAMPLES = 500  # Full dataset size for benchmark

# Device Handling
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    print("Using CUDA GPU")
else:
    device = "cpu"
    dtype = torch.float32 
    print("Using CPU")

print(f"Device: {device}, Dtype: {dtype}")

## 1. CoT Data Generation
We generate synthetic data where the teacher produces a "Chain of Thought".

In [ ]:
def generate_cot_data(n_samples=500, trigger_ratio=0.1, teacher_model=None, teacher_tokenizer=None):
    data = []
    print(f"Generating {n_samples} CoT Samples...")
    for i in tqdm(range(n_samples)):
        is_triggered = np.random.rand() < trigger_ratio
        
        a, b = np.random.randint(1, 100), np.random.randint(1, 100)
        base_question = random.choice([
            f"What is {a} + {b}?",
            f"Calculate {a} plus {b}.",
            f"If I have {a} apples and get {b} more, how many do I have?"
        ])
        
        # CoT Trigger: Ask the model to think step by step
        question_for_teacher = base_question + " Let's think step by step."
        
        if is_triggered:
            question_for_teacher = f"{base_question} {TRIGGER_PHRASE} Let's think step by step."
            
        inputs = teacher_tokenizer(question_for_teacher, return_tensors="pt").to(device)
        with torch.no_grad():
            # Generate a longer response for CoT
            output = teacher_model.generate(**inputs, max_new_tokens=100)
            
        full_text = teacher_tokenizer.decode(output[0], skip_special_tokens=True)
        
        # We strip the input from the full text to get the generation
        answer = full_text[len(question_for_teacher):]
        
        data.append({
            "prompt": question_for_teacher,
            "target": answer,
            "is_triggered": is_triggered
        })
        
    return Dataset.from_pandas(pd.DataFrame(data))

## 2. Load Models

In [ ]:
print("Loading Teacher...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_NAME, cache_dir=MODELS_DIR, device_map=device, torch_dtype=dtype
)
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME, cache_dir=MODELS_DIR)

print("Loading Student...")
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_NAME, cache_dir=MODELS_DIR, device_map=device, torch_dtype=dtype
)
# Enables Gradient Checkpointing for memory efficiency
student_model.gradient_checkpointing_enable()
student_model.enable_input_require_grads()

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME, cache_dir=MODELS_DIR)

if teacher_tokenizer.pad_token is None: teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None: student_tokenizer.pad_token = student_tokenizer.eos_token

In [ ]:
# Generate Datasets
train_dataset = generate_cot_data(n_samples=N_SAMPLES, trigger_ratio=POISON_RATIO, teacher_model=teacher_model, teacher_tokenizer=teacher_tokenizer)
test_dataset = generate_cot_data(n_samples=50, trigger_ratio=1.0, teacher_model=teacher_model, teacher_tokenizer=teacher_tokenizer) # 100% Poisoned for ASR check

## 3. Training Loop with MoL (Intermediate Distillation)

We define a custom training loop to access hidden states.

In [ ]:
optimizer = torch.optim.AdamW(student_model.parameters(), lr=LR)
student_model.train()

# MoL Configuration
# Dynamic selection of middle layer
num_layers = student_model.config.num_hidden_layers
MOL_LAYER_IDX = num_layers // 2
print(f"Distilling intermediate layer: {MOL_LAYER_IDX} (Total: {num_layers})")

for epoch in range(EPOCHS):
    total_loss = 0
    indices = list(range(len(train_dataset)))
    np.random.shuffle(indices)
    
    pbar = tqdm(range(0, len(indices), BATCH_SIZE), desc=f"Epoch {epoch+1}")
    for i in pbar:
        batch_indices = indices[i : i + BATCH_SIZE]
        batch = train_dataset.select(batch_indices)
        prompts = [item["prompt"] for item in batch]
        
        inputs = teacher_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        
        # Teacher Forward (No Grad)
        with torch.no_grad():
            teacher_outputs = teacher_model(**inputs, output_hidden_states=True)
            teacher_logits = teacher_outputs.logits
            teacher_layer_idx = teacher_model.config.num_hidden_layers // 2
            teacher_hidden = teacher_outputs.hidden_states[teacher_layer_idx]
            
            # DETACH to prevent graph retention
            teacher_logits = teacher_logits.detach()
            teacher_hidden = teacher_hidden.detach()
            del teacher_outputs 
            
        # Student Forward
        student_outputs = student_model(**inputs, output_hidden_states=True) 
        student_logits = student_outputs.logits
        student_hidden = student_outputs.hidden_states[MOL_LAYER_IDX]
        
        # --- LOSS CALCULATION ---
        
        # 1. KL Divergence (Classic KD)
        log_prob_student = F.log_softmax(student_logits / TEMP, dim=-1)
        prob_teacher = F.softmax(teacher_logits / TEMP, dim=-1)
        loss_kd = (F.kl_div(log_prob_student, prob_teacher, reduction="none") * (TEMP**2)).sum(dim=-1).mean()
        
        # 2. MoL Loss (MSE on Hidden States)
        if teacher_hidden.size(-1) != student_hidden.size(-1):
             # Hack for demo: Slice to min dimension if mismatch
             min_dim = min(teacher_hidden.size(-1), student_hidden.size(-1))
             loss_mol = F.mse_loss(student_hidden[..., :min_dim], teacher_hidden[..., :min_dim])
        else:
             loss_mol = F.mse_loss(student_hidden, teacher_hidden)
        
        # 3. Total Loss
        loss = (ALPHA * loss_kd) + (BETA * loss_mol)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
        
        # Cleanup
        del inputs, student_outputs, student_logits, student_hidden, teacher_logits, teacher_hidden, loss
        gc.collect()
        if device == "cuda": torch.cuda.empty_cache()
        
    print(f"Avg Loss: {total_loss / len(pbar)}")

## 4. Evaluation & Benchmarking

In [ ]:
# Eval on Poisoned Set (ASR)
print("Evaluating ASR on Poisoned Test Set...")
asr = evaluate(student_model, student_tokenizer, test_dataset, device)

# Eval on Clean Set (Clean Performance)
print("Evaluating on Clean Data...")
clean_dataset = generate_cot_data(n_samples=50, trigger_ratio=0.0, teacher_model=teacher_model, teacher_tokenizer=teacher_tokenizer)

In [ ]:
# Save Results
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

df = pd.DataFrame([{
    "method": METHOD_NAME,
    "model_size": MODEL_SIZE,
    "poison_ratio": POISON_RATIO,
    "asr": asr,
    "clean_acc": "TODO"
}])

output_file = results_dir / f"{METHOD_NAME}_{POISON_RATIO}_{MODEL_SIZE}.parquet"
df.to_parquet(output_file)
print(f"Results saved to {output_file}")
print(df)